# Advanced Retrieval with the Cat Health PDF

Session 1 used a dense vector retriever:

```text
question -> embed -> nearest chunks -> answer
```

This notebook keeps the same cat-health PDF and compares dense vector search, BM25, parent-child retrieval, hybrid retrieval, Cohere reranking, and multi-query retrieval.

> This is a retrieval exercise, not veterinary guidance. Answer only from retrieved context. Recommend a veterinarian for diagnosis, treatment, medication, or urgent-care decisions.


## Learning Outcomes

By the end of this session, you will be able to:

- Explain the different failure modes of dense and BM25 retrieval.
- Fuse independent ranked lists with reciprocal rank fusion (RRF).
- Increase recall with multiple generated search queries.
- Search focused child chunks while returning useful parent-page context.
- Use Cohere Rerank as a second-stage ranker.
- Compare retrieval systems with reviewed cases, visible evidence, metrics, and latency.


## Build Order

Build and compare the following layers:

1. Naive dense RAG: in-memory Qdrant, OpenAI embeddings, nearest child chunks, and a grounded answer.
2. BM25: sparse lexical retrieval over the same chunks.
3. Parent-child retrieval: search precise child chunks and return their parent PDF pages.
4. Hybrid retrieval with Cohere reranking: fuse dense and BM25 candidates with reciprocal rank fusion (RRF), recover parent pages, then rerank them.
5. Multi-query retrieval: generate alternate searches before the hybrid retrieve-then-rerank pipeline.

Dense retrieval plus BM25 is **hybrid retrieval** (or hybrid search). RRF is an **ensemble** method that combines their ranked lists. Adding Cohere reranking creates a **two-stage hybrid retrieve-then-rerank pipeline**.

Extra stages add latency, cost, and sometimes noise. Evaluate each added stage against those trade-offs.


## Task 1: Setup

Install the project environment from the session folder with `uv sync`, then run this notebook from that same folder. You need `OPENAI_API_KEY` and `COHERE_API_KEY` available when the notebook prompts for them.


In [1]:
from __future__ import annotations

import os
import re
from dataclasses import replace
from getpass import getpass
from pathlib import Path
from typing import Iterable, Sequence

import pandas as pd
from langchain_cohere import CohereRerank
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

from lib import (
    AnswerOutput,
    EvalCase,
    EvalItem,
    RetrievedDocument,
    answer_similarity_scorer,
    compare_eval_reports,
    compare_reports,
    faithfulness_scorer,
    make_openai_faithfulness_judge,
    run_eval,
    run_retrieval_eval,
)


/tmp/ipykernel_1880/2795770623.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Cohere API Key: ")

CHAT_MODEL = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")
EVAL_MODEL = os.environ.get("AIM_EVAL_MODEL", CHAT_MODEL)
EMBEDDING_MODEL = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")
RERANK_MODEL = os.environ.get("AIM_RERANK_MODEL", "rerank-v4.0-fast")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
answer_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
evaluation_model = ChatOpenAI(model=EVAL_MODEL, temperature=0)
query_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)

print(f"Chat model: {CHAT_MODEL}")
print(f"Evaluation model: {EVAL_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Cohere rerank model: {RERANK_MODEL}")


Chat model: gpt-5.4-mini
Evaluation model: gpt-5.4-mini
Embedding model: text-embedding-3-small
Cohere rerank model: rerank-v4.0-fast


## Task 2: Load and Chunk the Cat PDF for Naive RAG

Load the bundled PDF as page-level documents, then split each page into focused chunks for the dense RAG baseline. Preserve source and page metadata on every chunk.


In [3]:
pdf_path = Path("data/cat_health_guidelines.pdf")
if not pdf_path.exists():
    raise FileNotFoundError(f"Expected the cat PDF at {pdf_path.resolve()}")

pages = [page for page in PyPDFLoader(str(pdf_path)).load() if page.page_content.strip()]
for page_number, page in enumerate(pages, start=1):
    page.metadata.update(
        {
            "source": pdf_path.name,
            "page": page_number,
            "page_id": f"page-{page_number:02d}",
            "document_type": "cat_health_guideline",
        }
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")
print(pages[0].metadata)


Loaded 22 text-containing PDF pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline'}


In [4]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    add_start_index=True,
)
children = child_splitter.split_documents(pages)
for index, child in enumerate(children):
    child.metadata["chunk_id"] = f"child-{index:03d}"

print(f"Created {len(children)} child chunks from {len(pages)} parent pages.")
print(children[0].metadata)
print(children[0].page_content[:500])


Created 159 child chunks from 22 parent pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline', 'start_index': 0, 'chunk_id': 'child-000'}
VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are publishe


### ❓ Question 1: Traceability

Why should every child chunk retain its source file and page metadata?

Answer:

- Embedded chunks need to retain the source file metadata so that the generated answer can cite its sources and be verified. Without it, its impossible to check if the generated answer didn't hallucinate the retrieved source, and outdated or stale chunks can't be filtered out. Page metadata is needed to be able to retrieve the parent page as context, and to know which parent chunks (pages) were already retrieved, and deduplicate retrieved parents. 




### Shared Result Representation

All retrievers return the same RetrievedDocument type. Stable chunk and page IDs make the evidence inspectable and keep later comparisons fair.


In [5]:
def as_retrieved_document(document: Document, score: float | None = None) -> RetrievedDocument:
    return RetrievedDocument(
        id=document.metadata["chunk_id"],
        text=document.page_content,
        score=float(score) if score is not None else None,
        evidence_ids=(document.metadata["page_id"],),
        metadata=dict(document.metadata),
    )


def print_results(results: Sequence[RetrievedDocument], text_limit: int = 260) -> None:
    for rank, result in enumerate(results, start=1):
        page = result.metadata.get("page", "?")
        score = "n/a" if result.score is None else f"{result.score:.4f}"
        print(f"#{rank} | {result.id} | page={page} | score={score}")
        print(result.text[:text_limit].replace("\n", " "))
        print()


## Task 3: Naive Dense Vector RAG with In-Memory Qdrant

Session 1 baseline:

question -> OpenAI embedding -> Qdrant nearest-neighbor search -> answer

Create an in-memory Qdrant collection from the child chunks. Retrieve the nearest chunks, then pass only those chunks to the answer model. Use this **naive RAG baseline** for later comparisons.


In [9]:
# Qdrant stays in memory for this notebook. It disappears when the kernel stops.
vector_store = QdrantVectorStore.from_documents(
    documents=children,
    embedding=embeddings,
    location=":memory:",
    collection_name="cat_health_naive_dense_rag",
)

FIRST_STAGE_K = 8


def dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    matches = vector_store.similarity_search_with_score(question, k=k)
    return [as_retrieved_document(document, score) for document, score in matches]


dense_preview = dense_retrieve("What should a senior cat wellness visit cover?", k=4)
print_results(dense_preview)


#1 | child-034 | page=7 | score=0.6785
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with comorbidities. Speci ﬁc questions regarding changes in appet

#2 | child-021 | page=6 | score=0.6741
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care re

#3 | child-060 | page=10 | score=0.6187
study, three 10- to 15-minute exercise sessions per day led to a loss of approximately 1% of body weight in 1 month with no food intake restrictions. 66 Senior Cats Senior cats exhibiting new or unusual behavior should be evaluated for medical conditions. 12 C

#4 | child-036 | page=8 | score=0.6184
Detecting signs of pain or anxiety and evaluation of qual

### Naive RAG Answer

Retrieval quality and answer quality are separate concerns. Pass the nearest dense chunks to the answer model. The same grounded-answer function will later receive context from the other retrievers.


In [20]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer only from the provided cat-health guideline context. "
            "Do not diagnose, prescribe, or make an urgent-care decision. "
            "If the context is insufficient, say so. End with a short Sources line.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)


def format_context(documents: Sequence[RetrievedDocument]) -> str:
    blocks = []
    for document in documents:
        page = document.metadata.get("page", "unknown")
        blocks.append(f"[source={document.id}; page={page}]\n{document.text}")
    return "\n\n".join(blocks)


def answer_with_retriever(
    question: str,
    retriever,
    k: int = 4,
) -> AnswerOutput:
    documents = retriever(question, k=k)
    response = answer_model.invoke(
        answer_prompt.format_messages(question=question, context=format_context(documents))
    )
    return AnswerOutput(answer=str(response.content), documents=tuple(documents))


naive_result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=dense_retrieve,
)
print(naive_result.answer)
print("\nNaive dense evidence:")
print_results(naive_result.documents, text_limit=200)


At a senior cat wellness visit, an owner should discuss any **changes in appetite**, **urination or drinking** (polyuria/polydipsia), **vomiting or hairballs**, **diarrhea**, **litter box usage**, **new or unusual behavior**, **increased nighttime activity or vocalization**, and **changes in normal habits or activity**. These may help the veterinarian look for disease, pain, mobility changes, vision or hearing changes, or cognitive dysfunction.

Sources: child-034 p.7; child-060 p.10; child-021 p.6

Naive dense evidence:
#1 | child-034 | page=7 | score=0.6950
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with 

#2 | child-021 | page=6 | score=0.6693
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines ar

## Task 4: BM25 Sparse Retrieval

BM25 ranks the same child chunks with lexical term matches rather than embeddings. It is useful for abbreviations, age ranges, named conditions, and phrases that dense similarity may blur.

Keep the corpus, chunks, and retrieval depth fixed. The retriever is the only thing that changes.


In [13]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(child.page_content) for child in children])


def bm25_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    scores = bm25.get_scores(tokenize(question))
    ranked_indices = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
    return [
        as_retrieved_document(children[index], float(scores[index]))
        for index in ranked_indices[:k]
    ]


bm25_preview = bm25_retrieve("What do BCS and MCS stand for?", k=4)
print_results(bm25_preview)


#1 | child-079 | page=13 | score=11.1440
could result in health problems. 83 No matter the life stage, to help avoid potential nutrient insufﬁciencies, cats should be fed diets labeled with an Association of American Feed Control Of ﬁcials statement of nutritional adequacy. AAHA and the AAFP do not a

#2 | child-029 | page=6 | score=10.8317
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#3 | child-006 | page=1 | score=9.7786
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus); FHV-1 (feline herpesvirus type 1); FIC (feline idiopathic cystitis); FPV (feline panleukopenia virus); GI (ga

#4 | child-082 | page=13 | score=9.7514
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus S

### ❓ Question 2: When Should BM25 Win?

Which query is more likely to favor BM25: `What do BCS and MCS stand for?` or `How should an owner get ready for a senior-cat wellness visit?` Why?

Answer:
- The query `What do BCS and MCS stand for?` is more likely to favor BM25 because it holds rare keyword specific terms. BM25 is based on IDF (Inverse Document Frequency) and TF (term frequency)  Chunks which hold rare keywords like BCS and MCS will be scored higher. Also, arconyms like 'BCS'/'MCS' are terms with little semantic content on their own. An embedding model, especially a more simple one, will struggle placing them meaningfully in an embedding space.
- The second query doesn't hold rare keywords which are semantically meaninful when isolated, making dense retrieval more effective, which retrieves vectors embedding entire chunks most similar semantically to the whole query. 

### 🚀 Activity 1: Dense vs. BM25 Failure Modes

Pick three questions: one paraphrase-heavy question, one exact-term question, and one broad multi-part question. Inspect both result lists before generating an answer.

- Which retriever put the best evidence first?
- Which retriever returned redundant chunks?
- Which question type should favor BM25, and why?

Answer:

- For query "What does the guideline say about BCS and MCS?":
    - BM25 retrieved first 2 chunks which seemed to be equaly highly relevant, (#1 ranked child-005 mentions they are important components of the physical examination at all life stages, #2 child-082 mentions energy requirements of cats are in ﬂuenced by them). The third chunk was redundant (it holds the term BCS but out of context).
    - Dense retrieved #1 and #3 chunks were redundant, and #2 chunk was the same child-005 relevevant chunk. 
        - Conclusion for this exact term question, BM25 put the best evidence first, as expected following the answer to question #1, which concludes that exact-term queries favor BM25.
 

- For query "How often should senior cats see a veterinarian?":
    - Dense retrieved #1 child-021 was highly relevant, and #2 child-108 chunk was partially relevant were redundant, third chunk was redundant.         
    - BM25 retrieved #1 and #3 chunks were redundant, and #2 chunk was the same child-108 relevevant chunk. 
        - Conclusion for this paraphrase question, Dense retrieval managed to semantically both retrieve chunks which mention frequency visits (paraphrase of often) and did it first. This question is heavy on semantics, eactly where dense retrieval should shine.     

- For query "What factors shape an individualized preventive-care plan?":
    - Both BM25 and Dense retrieved #1 child-023 was relevant (though generally), and for both retrieval methods chunks 2 and 3 were redundant.
    - This result makes sense, as both retrieval methods are limited with this question, which is both abstract/multipart, needing an answer spread across different chunks, and niether dense retrieval nor BM25 connect context between different chunks. Also, it doesn't hold rare specific terms for BM25 to be effective.  

In [11]:
activity_questions = [
    "What does the guideline say about BCS and MCS?",
    "How often should senior cats see a veterinarian?",
    "What factors shape an individualized preventive-care plan?",
]

for question in activity_questions:
    print(f"\nQUESTION: {question}\n")
    print("DENSE")
    print_results(dense_retrieve(question, k=3), text_limit=800) #removed the text limit to inspect the results
    print("BM25")
    print_results(bm25_retrieve(question, k=3), text_limit=800)



QUESTION: What does the guideline say about BCS and MCS?

DENSE
#1 | child-005 | page=1 | score=0.4298
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted based on the needs of the individual patient, resources, and limitations unique to each individual practice setting. Evidence-based support for speciﬁc recommendations has been cited whenever pos- sible and appropriate. Other recommendations are based on practical clinical experience and a consensus of expert opinion. Further research is needed to document some of these recommendations. Because each case is different, vet- erinarians must base their decisions on the best available scientiﬁc ev- idence in conjunction with their own knowledge and experience. BCS (body condition score); DER (daily energy requirements); DJD

#2 | child-029 | page=6 | score=0.4141
Evaluation and recording of body weight, body condition score (BCS), and muscle 

## Task 5: Parent-Child Retrieval

Build a parent-child pipeline on top of the dense retriever:

question -> dense child search in Qdrant -> matching parent PDF pages

Child chunks give the vector store a focused search surface. Parent pages give the answer model surrounding context. Parent-child retrieval adds context recovery to dense retrieval; it is not hybrid retrieval.


In [11]:
# Parent-page lookup begins here; naive RAG does not need parent records.
parents_by_id = {page.metadata["page_id"]: page for page in pages}


def recover_parent_documents(
    child_candidates: Sequence[RetrievedDocument],
    k: int,
) -> list[RetrievedDocument]:
    parents: list[RetrievedDocument] = []
    seen_parent_ids: set[str] = set()

    for child in child_candidates:
        page_id = child.metadata["page_id"]
        if page_id in seen_parent_ids:
            continue
        parent = parents_by_id[page_id]
        parents.append(
            RetrievedDocument(
                id=page_id,
                text=parent.page_content,
                score=child.score,
                evidence_ids=(page_id,),
                metadata={
                    **parent.metadata,
                    "retrieved_from_child": child.id,
                    "first_stage_score": child.score,
                },
            )
        )
        seen_parent_ids.add(page_id)
        if len(parents) == k:
            break
    return parents


def parent_child_dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = dense_retrieve(question, k=FIRST_STAGE_K)
    return recover_parent_documents(child_candidates, k=k)


parent_preview = parent_child_dense_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(parent_preview, text_limit=400)


#1 | page-06 | page=6 | score=0.6861
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-07 | page=7 | score=0.6549
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require additional focus during the examination by each life stage are listed in Table 3. Kittens Kittens will have different health risks dependin

#3 | page-10 | page=10 | score=0.6433
including carpeting, window and door frames, curtains, and couches. Keeping the nail

### ❓ Question 3: Search Small, Return Large?

Why does parent-child retrieval search focused child chunks but return the larger parent pages instead of indexing and returning only full pages?

Answer:

- Indexing retrieving only full pages means that each chunk is the size of a full pdf page. As we learned in session 1, when a chunk size is too big, it adds irrelevant data relative to a query, averaging out its relevance scoring when retrieving the top chunks. This can cause irrelavnt pages to be retrieved. Also, when tracing the result, not having a focused chunk  will make it harder to verify that the retrieved page is relevant to the query.
- When searching focused chunks and returning the parent page, the precision of retrieval can be gained, together with the completeness in context of the parent page, as the child chunk might be too fragmented to answer the query fully on its own, requireing the surrounding context of the rest of the page.


# Breakout Room #2: Combine, Rank, and Expand

The first half produced three building blocks: dense retrieval, BM25, and parent-child context recovery. This breakout combines them and compares the results.

In this breakout:

1. Fuse dense and BM25 rankings with reciprocal rank fusion.
2. Rerank the broad hybrid candidate set with Cohere.
3. Expand vague questions into multiple searches.
4. Compare the trade-offs with the local evaluation library.

Keep only the stages that improve measured retrieval enough to justify their latency.


## Task 6: Hybrid Retrieval — Dense + BM25

Dense and BM25 scores use different scales, so raw scores cannot be added directly. Reciprocal rank fusion (RRF) works from their **ranked lists** instead.

The hybrid first stage is:

dense candidates + BM25 candidates -> RRF -> hybrid child candidates

**Hybrid retrieval** combines semantic vector retrieval with sparse lexical retrieval. RRF makes it an **ensemble retriever** by fusing multiple rankings.


In [14]:
def reciprocal_rank_fusion(
    ranked_lists: Iterable[Sequence[RetrievedDocument]],
    *,
    limit: int,
    rrf_constant: int = 60,
) -> list[RetrievedDocument]:
    scores: dict[str, float] = {}
    documents_by_id: dict[str, RetrievedDocument] = {}

    for ranked_list in ranked_lists:
        for rank, document in enumerate(ranked_list, start=1):
            documents_by_id.setdefault(document.id, document)
            scores[document.id] = scores.get(document.id, 0.0) + 1 / (rrf_constant + rank)

    return [
        replace(
            documents_by_id[document_id],
            score=score,
            metadata={
                **documents_by_id[document_id].metadata,
                "rrf_score": score,
            },
        )
        for document_id, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)[:limit]
    ]


def hybrid_children_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    return reciprocal_rank_fusion(
        [
            dense_retrieve(question, k=FIRST_STAGE_K),
            bm25_retrieve(question, k=FIRST_STAGE_K),
        ],
        limit=k,
    )


hybrid_preview = hybrid_children_retrieve("What does the guideline say about BCS and MCS?", k=4)
print_results(hybrid_preview)


#1 | child-029 | page=6 | score=0.0325
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#2 | child-030 | page=7 | score=0.0310
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require

#3 | child-005 | page=1 | score=0.0164
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted based on the needs of the individual patient, resources, and limitations unique to each individual practice sett

#4 | child-082 | page=13 | score=0.0161
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus Stat

## Task 7: Cohere Reranking over Hybrid Candidates

Use hybrid retrieval to gather a broad set of plausible child chunks from dense and BM25 search. Recover their parent pages, then send those candidates and the question to Cohere Rerank.

dense + BM25 -> RRF -> parent pages -> Cohere Rerank -> final context

The result is a **two-stage hybrid retrieve-then-rerank pipeline**. Cohere scores rank documents within one query and candidate set; do not treat them as universal probabilities.

The custom RRF function supplies the candidate list, so the notebook calls LangChain's `CohereRerank` compressor directly. A single LangChain `BaseRetriever` could instead be wrapped with `ContextualCompressionRetriever`.


In [15]:
RERANK_CANDIDATE_K = 8


def to_langchain_document(candidate: RetrievedDocument) -> Document:
    return Document(
        page_content=candidate.text,
        metadata={
            **candidate.metadata,
            "retrieved_id": candidate.id,
            "evidence_ids": list(candidate.canonical_evidence_ids),
            "first_stage_score": candidate.score,
        },
    )


def to_reranked_document(document: Document) -> RetrievedDocument:
    relevance_score = document.metadata.get("relevance_score")
    return RetrievedDocument(
        id=document.metadata["retrieved_id"],
        text=document.page_content,
        score=float(relevance_score) if relevance_score is not None else None,
        evidence_ids=tuple(document.metadata["evidence_ids"]),
        metadata=dict(document.metadata),
    )


def rerank_parent_candidates(
    question: str, candidates: Sequence[RetrievedDocument], k: int
) -> list[RetrievedDocument]:
    compressor = CohereRerank(model=RERANK_MODEL, top_n=k)
    reranked_documents = compressor.compress_documents(
        documents=[to_langchain_document(candidate) for candidate in candidates],
        query=question,
    )
    return [to_reranked_document(document) for document in reranked_documents]


def hybrid_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = hybrid_children_retrieve(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


rerank_preview = hybrid_reranked_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(rerank_preview, text_limit=400)


#1 | page-06 | page=6 | score=0.7716
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-18 | page=18 | score=0.7030
events to increase knowledge and con ﬁdence when taking patient histories (see “Conducting Effective Patient Histories” box) and pro- viding client education are just as important as further education on feline-friendly handling, disease processes, and technical skills. Ideally, client education is a key responsibility for all staff members. Every life stage will have speci ﬁc items that should be

#3 | page-10 | page=10 | score=0.6923
including carpeting, window and door frames, curtains, and couches. Keeping the nai

## Task 8: Multi-Query Retrieval

A user question may be vague, incomplete, or phrased differently from the source. Generate alternate search queries, run each through hybrid first-stage retrieval, and fuse the resulting child rankings.

Multi-query retrieval expands recall on top of the hybrid retrieve-then-rerank pipeline. It also adds model calls, latency, and possible noise.


In [16]:
multi_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write {count} short, distinct search queries for the cat health guideline PDF. "
            "Return one query per line. Do not answer the question.",
        ),
        ("human", "{question}"),
    ]
)


def generate_query_variants(question: str, count: int = 3) -> list[str]:
    response = query_model.invoke(
        multi_query_prompt.format_messages(question=question, count=count)
    )
    variants = [question]
    for line in response.content.splitlines():
        candidate = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
        if candidate and candidate.lower() not in {item.lower() for item in variants}:
            variants.append(candidate)
    return variants[: count + 1]


def multi_query_hybrid_children(question: str, k: int = 5) -> list[RetrievedDocument]:
    ranked_lists: list[Sequence[RetrievedDocument]] = []
    for variant in generate_query_variants(question):
        ranked_lists.append(dense_retrieve(variant, k=FIRST_STAGE_K))
        ranked_lists.append(bm25_retrieve(variant, k=FIRST_STAGE_K))
    return reciprocal_rank_fusion(ranked_lists, limit=k)


def multi_query_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = multi_query_hybrid_children(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


question = "How should an owner prepare for a senior cat wellness visit?"
print(generate_query_variants(question))
print_results(multi_query_reranked_retrieve(question, k=4), text_limit=400)


['How should an owner prepare for a senior cat wellness visit?', 'senior cat wellness visit preparation PDF', 'cat health guideline senior wellness exam checklist PDF', 'owner preparation for senior cat checkup PDF']
#1 | page-06 | page=6 | score=0.7716
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-18 | page=18 | score=0.7030
events to increase knowledge and con ﬁdence when taking patient histories (see “Conducting Effective Patient Histories” box) and pro- viding client education are just as important as further education on feline-friendly handling, disease processes, and technical skills. Ideally, client education is a key 

## Task 9: Compare Retrieval and Answer Results

The local library measures retrieval and answer quality separately.

- **Retrieval:** Hit@k, Precision@k, Recall@k, MRR, and latency.
- **Faithfulness:** the share of answer statements supported by the passages retrieved for that answer. The OpenAI judge records a statement, reason, and 0/1 verdict for each claim.
- **Answer similarity:** cosine similarity between OpenAI embeddings of the generated answer and a reviewed reference answer.

First, compare five retrieval pipelines on the same reviewed text-retrieval cases:

1. Naive dense RAG.
2. BM25.
3. Dense parent-child retrieval.
4. Hybrid dense + BM25 retrieval with Cohere reranking.
5. Multi-query hybrid retrieve-then-rerank.

Inspect per-case rankings alongside aggregate metrics before choosing a system. The visual life-stage table needs separate handling: the PDF text extractor loses most of its cells.


In [17]:
text_reviewed_cases = [
    EvalCase(
        id="life-stage-definitions",
        query="What feline life stages does the guideline define?",
        relevant_evidence_ids=("page-03",),
        tags=("baseline", "life-stage"),
    ),
    EvalCase(
        id="senior-visit-frequency",
        query="How often should senior cats be seen by a veterinarian?",
        relevant_evidence_ids=("page-06",),
        tags=("exact", "senior"),
    ),
    EvalCase(
        id="bcs-mcs",
        query="What do BCS and MCS mean, and why are they recorded?",
        relevant_evidence_ids=("page-06", "page-07"),
        tags=("acronym", "dense-vs-bm25"),
    ),
]

table_layout_challenge = EvalCase(
    id="life-stage-table",
    query="How do wellness discussion items change across feline life stages?",
    relevant_evidence_ids=("page-04", "page-05"),
    tags=("table", "parent-child", "requires-visual-review"),
    notes="Use this after adding a PDF-page vision fallback.",
)

EVAL_K = 4
dense_report = run_retrieval_eval("naive_dense", text_reviewed_cases, dense_retrieve, k=EVAL_K)
bm25_report = run_retrieval_eval("bm25", text_reviewed_cases, bm25_retrieve, k=EVAL_K)
parent_child_report = run_retrieval_eval(
    "dense_parent_child",
    text_reviewed_cases,
    parent_child_dense_retrieve,
    k=EVAL_K,
)
hybrid_rerank_report = run_retrieval_eval(
    "hybrid_rrf_then_cohere",
    text_reviewed_cases,
    hybrid_reranked_retrieve,
    k=EVAL_K,
)
multi_query_report = run_retrieval_eval(
    "multi_query_hybrid_rerank",
    text_reviewed_cases,
    multi_query_reranked_retrieve,
    k=EVAL_K,
)

comparison = compare_reports(
    dense_report,
    bm25_report,
    parent_child_report,
    hybrid_rerank_report,
    multi_query_report,
)
comparison


,retriever,k,cases,hit_rate,precision_at_k,recall_at_k,mrr,mean_latency_ms
0,dense_parent_child,4,3,1.0,0.333333,1.0,1.000000,163.653876
1,naive_dense,4,3,1.0,0.333333,1.0,1.000000,531.833258
2,hybrid_rrf_then_cohere,4,3,1.0,0.333333,1.0,0.777778,472.758116
3,multi_query_hybrid_rerank,4,3,1.0,0.333333,1.0,0.750000,2467.894524
4,bm25,4,3,1.0,0.333333,1.0,0.583333,0.359697


In [18]:
multi_query_report.case_table()


,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[page-18, page-01, page-02, page-03]",1.0,0.25,1.0,0.25,3298.753738
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[page-06, page-02, page-10, page-16]",1.0,0.25,1.0,1.00,1867.584844
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[page-06, page-13, page-01, page-07]",1.0,0.50,1.0,1.00,2237.344991


### Answer-Level Evaluation: Faithfulness and Answer Similarity

Retrieval success does not guarantee that an answer stays grounded in its retrieved passages or covers the reviewed answer. The cases below add reviewed reference answers.

The generic runner follows the same shape as Evalite: data, task, and scorers. This comparison uses the naive dense baseline and the complete multi-query pipeline. Each answer is scored against its own retrieved passages for faithfulness, then against the same reviewed reference answer for semantic similarity.


In [ ]:
answer_reviewed_cases = [
    EvalItem(
        id="life-stage-definitions",
        input="What feline life stages does the guideline define?",
        expected=(
            "The guidelines define kitten (birth to 1 year), young adult (1 through 6 years), "
            "mature adult (7 through 10 years), and senior (over 10 years). End of life can occur at any age."
        ),
        tags=("life-stage",),
    ),
    EvalItem(
        id="senior-visit-frequency",
        input="How often should senior cats be seen by a veterinarian?",
        expected=(
            "All cats should have at least annual examinations. Senior cats should be seen at least every "
            "6 months, with more frequent visits for chronic conditions."
        ),
        tags=("senior",),
    ),
    EvalItem(
        id="bcs-mcs",
        input="What do BCS and MCS mean, and why are they recorded?",
        expected=(
            "BCS is body condition score and MCS is muscle condition score. Along with body weight, "
            "they should be evaluated and recorded at every life stage to identify changes and trends early."
        ),
        tags=("acronym",),
    ),
]

faithfulness_judge = make_openai_faithfulness_judge(evaluation_model)
answer_scorers = [
    faithfulness_scorer(faithfulness_judge),
    answer_similarity_scorer(embeddings),
]


def answerer_for(retriever):
    return lambda question: answer_with_retriever(question, retriever)





dense_answer_report = run_eval(
    "naive_dense_answer",
    data=answer_reviewed_cases,
    task=answerer_for(dense_retrieve),
    scorers=answer_scorers,
)



full_pipeline_answer_report = run_eval(
    "multi_query_hybrid_rerank_answer",
    data=answer_reviewed_cases,
    task=answerer_for(multi_query_reranked_retrieve),
    scorers=answer_scorers,
)



answer_comparison = compare_eval_reports(
    dense_answer_report,
    full_pipeline_answer_report,
)
answer_comparison


,evaluation,cases,faithfulness,answer_similarity,mean_task_latency_ms,mean_scoring_latency_ms
0,naive_dense_answer,3,1.000000,0.843468,1577.908053,1781.28262
1,multi_query_hybrid_rerank_answer,3,0.933333,0.862550,2902.509618,1960.66787


In [22]:
full_pipeline_answer_report.case_table()


,case_id,input,expected,tags,output,task_latency_ms,scoring_latency_ms,faithfulness,faithfulness_metadata,answer_similarity,answer_similarity_metadata
0,life-stage-definitions,What feline life stages does the guideline def...,The guidelines define kitten (birth to 1 year)...,life-stage,AnswerOutput(answer='The guideline defines fiv...,2802.781338,2443.257773,1.0,{'verdicts': [{'statement': 'The guideline def...,0.861722,{'raw_cosine_similarity': 0.8617222617859789}
1,senior-visit-frequency,How often should senior cats be seen by a vete...,All cats should have at least annual examinati...,senior,AnswerOutput(answer='Senior cats should be see...,2841.253101,1284.181827,1.0,{'verdicts': [{'statement': 'Senior cats shoul...,0.830842,{'raw_cosine_similarity': 0.830842216138726}
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...",BCS is body condition score and MCS is muscle ...,acronym,AnswerOutput(answer='BCS means **body conditio...,3063.494415,2154.564011,0.8,{'verdicts': [{'statement': 'BCS means body co...,0.895085,{'raw_cosine_similarity': 0.8950854361799626}


### ❓ Question 4: Is More Retrieval Always Better?

Suppose multi-query plus reranking improves recall but lowers MRR and adds noticeable latency. How would faithfulness and answer similarity change your decision about shipping it for every cat-health question?

Answer:
- A lower MMR means that the first relevant chunk was ranked lower than using another retrieval method. This means that possibly noise in total was added. But if recall improved, then we can definitely state that in total, more distinct relevant pages were retrieved (regardless of noise).
    - If Faithfulness and answer similarity also improve, this should mean that the added noise was negligible compared with the added relevant context, and the result is a more complete and grounded answer which should answer the original query better.
    - If faithfulness improved but answer similarty degraded, this means that despite the better recall, too much noise was added which caused the answer to degrade in context.
    - If faithfulness degraded and answer similarity improved, this would be a warning sign, as the answer would be "right for the wrong reason", and its possible the llm managed the answer the original query correctly despite relying less on the retrieved context. For a health assistent ungrounded answers can be dangerous, so I wouldn't ship the product with this result.  
For the happy path, the question is, by how much the faithfulness and answer similarity improved. If the improvement is negligible, then a noticable latency addition should tip the scale to ship the more naive retrieval approach. If reviewing the evaluation deeper shows that the "multi-query plus reranking" approach improves faithfulness And answer similarity mostly in a subset of questions (for examply only for abstract-multi-part questions), I would add a query identifier and route the query to the appropriate retieval. 



### 🚀 Activity 2: Make and Defend a Retrieval Recommendation

Choose one retrieval result and one answer-evaluation result, then make a recommendation for the cat-health application.

Include:

1. Which rung of the retrieval ladder you chose.
2. Evidence from at least two metrics and one inspected ranking or claim-level verdict.
3. Its cost/latency trade-off.
4. One case where you would choose a different rung instead.

A higher aggregate metric does not settle the product decision.


Answer: 

1. I chose the parent_child dense retrieval.
2.  Although all methods scored the same in 'hit_rate', 'precision_at_k' and 'recall_at_k', this retrieval method when used by the llm to answer the query, had 1.0 faithfulness [see helper table below], together with naive dense, and 0.885 answer_similarity, which was the highest similarity to the reference among all retrieval methods. The only way to trace and verify the faithfulness score was to print out the claims by the scorer, read its verdicts, and then check myself the cited sources.

The result: for the query: "What feline life stages does the guideline define?" The judge correctly verified all the claims which do appear in page 3 of the pdf. 

The generated answer was:
"The guideline defines these feline life stages:

- Kitten: birth to 1 year
- Young adult: 1 to 6 years
- Mature adult: 7 to 10 years
- Senior: over 10 years
- End-of-life: can occur at any age

It also notes that the guidelines focus on kitten through senior. Sources: page 3, page 2, page 1."

The judge verdicts were:

  [1] The guideline defines these feline life stages: Kitten, young adult, mature adult, senior, and end-of-life.
       reason: Supported by the passages, which state the Task Force designated four age-related life stages—kitten, young adult, mature adult, and senior—with a fifth end-of-life stage that can occur at any age.
  [1] Kitten is from birth to 1 year.
       reason: Directly supported by Source 1, which defines the kitten stage as from birth up to 1 year.
  [1] Young adult is from 1 to 6 years.
       reason: Directly supported by Source 1, which defines young adult as from 1 year through 6 years.
  [1] Mature adult is from 7 to 10 years.
       reason: Directly supported by Source 1, which defines mature adult as from 7 to 10 years.
  [1] Senior is over 10 years.
       reason: Directly supported by Source 1, which defines senior as aged over 10 years.
  [1] End-of-life can occur at any age.
       reason: Directly supported by Source 1, which states the fifth, end-of-life stage can occur at any age.
  [1] The guidelines focus on kitten through senior.
       reason: Directly supported by Source 1, which says the guidelines focus on the life stages of kitten through to senior.

---

from page 3: "The T ask Force has designated four age-related life stages (T able 1): the
kitten stage, from birth up to 1 year; young adult, from 1 year through
6 years; mature adult, from 7 to 10 years; and senior, aged over 10
years. The ﬁfth, end-of-life stage can occur at any age. These guide-
lines focus on the life stages of kitten through to senior."


3. cost/latency tradeoff: The parent_child dense retrieval task latency was 1336ms, as opposed to multi_query_hybrid which is 2902ms, and on par with hybrid_rrf_then_cohere (1306) and dense naive (1577ms). It had significantly more latency than bm25 (952ms). Although the metric results in this comparison showed that all retrievers performed similarly, we do know bm25 should struggle with queries holding semantic meaning instead of exact specific keywords, and the similar results are of low confidence due to the low sample size. The parent-child dense retrieval, provides an obvious advantage over naive-dense with context completeness, and the current evaluation is not comprehensive enough to show the real benefits of cohere-rerank and multi-query. Cohere rerank costs an additional model call and multi-query-hybrid costs another additional model call for multiple query generations, and extra embeddings, which is why I would recommend the parent_child dense retrieval for this application.

4. If a typical user would query a mixed variation of semantic phrasing and rare-exact terms, I'd use a hybrid retrieval with rerank, which would provide balance for retrieval of chunks holding exact terms at the cost of an additional cohere rerank model call.


In [25]:
dense_parent_child_answer_report = run_eval(          # added dense_parent_child retriever
    "dense_parent_child_answer",
    data=answer_reviewed_cases,
    task=answerer_for(parent_child_dense_retrieve),
    scorers=answer_scorers,
)

hybrid_rrf_then_cohere_answer_report = run_eval(   # added hybrid_rrf_then_cohere retriever
    "hybrid_rrf_then_cohere_answer",
    data=answer_reviewed_cases,
    task=answerer_for(hybrid_reranked_retrieve),
    scorers=answer_scorers,
)

bm25_answer_report = run_eval(         #added bm25 retriever
    "bm25_answer",
    data=answer_reviewed_cases,
    task=answerer_for(bm25_retrieve),
    scorers=answer_scorers,
)

answer_comparison = compare_eval_reports(
    dense_parent_child_answer_report,
    dense_answer_report,
    hybrid_rrf_then_cohere_answer_report,
    full_pipeline_answer_report,
    bm25_answer_report,
)
answer_comparison

,evaluation,cases,faithfulness,answer_similarity,mean_task_latency_ms,mean_scoring_latency_ms
0,dense_parent_child_answer,3,1.000000,0.885828,1336.027259,2411.799653
1,naive_dense_answer,3,1.000000,0.843468,1577.908053,1781.282620
2,bm25_answer,3,1.000000,0.839816,952.921951,1710.116604
3,multi_query_hybrid_rerank_answer,3,0.933333,0.862550,2902.509618,1960.667870
4,hybrid_rrf_then_cohere_answer,3,0.916667,0.848484,1306.021319,1729.365147


In [26]:
parent_child_report.case_table()


,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[page-03, page-07, page-02, page-01]",1.0,0.25,1.0,1.0,174.086483
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[page-06, page-16, page-07, page-10]",1.0,0.25,1.0,1.0,166.490913
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[page-06, page-01, page-07, page-19]",1.0,0.50,1.0,1.0,150.384233


In [34]:
dense_parent_child_answer_report.case_table()


,case_id,input,expected,tags,output,task_latency_ms,scoring_latency_ms,faithfulness,faithfulness_metadata,answer_similarity,answer_similarity_metadata
0,life-stage-definitions,What feline life stages does the guideline def...,The guidelines define kitten (birth to 1 year)...,life-stage,AnswerOutput(answer='The guideline defines the...,2005.431350,3179.330407,1.0,{'verdicts': [{'statement': 'The guideline def...,0.908107,{'raw_cosine_similarity': 0.9081074137580832}
1,senior-visit-frequency,How often should senior cats be seen by a vete...,All cats should have at least annual examinati...,senior,AnswerOutput(answer='Senior cats should be see...,932.229231,1947.100251,1.0,{'verdicts': [{'statement': 'Senior cats shoul...,0.846394,{'raw_cosine_similarity': 0.8463935547046935}
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...",BCS is body condition score and MCS is muscle ...,acronym,AnswerOutput(answer='BCS means **body conditio...,1070.421196,2108.968302,1.0,{'verdicts': [{'statement': 'BCS means body co...,0.902982,{'raw_cosine_similarity': 0.9029819162793782}


In [45]:
report = dense_parent_child_answer_report
for row in report.rows:
    print(f"\n{'='*70}\nCASE: {row.item.id}   faithfulness={row.score('faithfulness').score}")
    print(f"\nGENERATED ANSWER:\n{row.output.answer}")
    print("JUDGE VERDICTS:")
    for v in row.score('faithfulness').metadata['verdicts']:
        print(f"  [{v['verdict']}] {v['statement']}\n       reason: {v['reason']}")      
    print("\nRETRIEVED SOURCES:")
    for d in row.output.documents:
        print(f"  [{d.id} | page={d.metadata.get('page')}]\n  {d.text[:300000]}\n")



CASE: life-stage-definitions   faithfulness=1.0

GENERATED ANSWER:
The guideline defines these feline life stages:

- Kitten: birth to 1 year
- Young adult: 1 to 6 years
- Mature adult: 7 to 10 years
- Senior: over 10 years
- End-of-life: can occur at any age

It also notes that the guidelines focus on kitten through senior. Sources: page 3, page 2, page 1.
JUDGE VERDICTS:
  [1] The guideline defines these feline life stages: Kitten, young adult, mature adult, senior, and end-of-life.
       reason: Supported by the passages, which state the Task Force designated four age-related life stages—kitten, young adult, mature adult, and senior—with a fifth end-of-life stage that can occur at any age.
  [1] Kitten is from birth to 1 year.
       reason: Directly supported by Source 1, which defines the kitten stage as from birth up to 1 year.
  [1] Young adult is from 1 to 6 years.
       reason: Directly supported by Source 1, which defines young adult as from 1 year through 6 years.
  [1] M

## Task 10: Answer with a Selected Pipeline

Pass only the selected pipeline's retrieved context to the answer model. Source labels keep the evidence inspectable. For the two-page life-stage table, inspect the original PDF page when text extraction does not preserve the table structure.


In [46]:
result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=parent_child_dense_retrieve,
)
print(result.answer)
print("\nRetrieved evidence:")
print_results(result.documents, text_limit=200) 



At a senior cat wellness visit, the owner should discuss:
- Any changes in appetite
- Increased urination or thirst
- Vomiting, hairball vomiting, or diarrhea
- Increased nighttime activity or vocalization
- Changes in the cat’s normal habits or activity level

These topics help the veterinarian look for possible changes in health, mobility, pain, vision, or cognitive function. If the context is needed for more detail, it is insufficient to go beyond these items.

Sources: page-06, page-07, page-10

Retrieved evidence:
#1 | page-07 | page=7 | score=0.6950
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner re

#2 | page-06 | page=6 | score=0.6693
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are inte

## Advanced Builds

- Add maximal marginal relevance (MMR) and measure whether it reduces redundant chunks.
- Add HyDE: generate a hypothetical answer, embed it, and use it as an alternate dense query.
- Add a PDF-page vision fallback for the life-stage table challenge.
- Add a second reviewed corpus and use metadata routing to avoid mixing evidence sets.

## Recap

Build in this order:

1. Start with a transparent naive dense RAG baseline in in-memory Qdrant.
2. Add BM25 to see the value of lexical retrieval.
3. Add parent-child recovery to improve answer context.
4. Combine dense and BM25 with RRF: hybrid / ensemble retrieval.
5. Rerank the hybrid parent candidates with Cohere: a two-stage retrieve-then-rerank pipeline.
6. Add multi-query expansion only when its recall benefit justifies its extra latency and cost.

Use the local evaluation library to inspect the retrieved evidence behind every aggregate number.
